# Análisis Comparativo: Modelo Original vs Modelo Optimizado
## S&P 500 Stock Analysis — Catriel Arandiga

Este notebook documenta el proceso de revisión crítica del modelo original de predicción de precios de acciones del S&P 500,
las mejoras implementadas, y los nuevos insights generados.

---

### Resumen ejecutivo

| Dimensión | Modelo Original | Modelo Optimizado |
|-----------|----------------|------------------|
| Target | Precio de apertura (mismo período) | Retorno forward a 3 meses |
| Data Leakage | Sí — close/high/low del mismo mes | No — fundamentales del año anterior + lags |
| Validación | KFold (mezcla pasado y futuro) | TimeSeriesSplit (respeta tiempo) |
| R² reportado | **0.9999** (inflado por leakage) | **-0.1059** (honesto) |
| Modelo | Random Forest | Gradient Boosting optimizado |
| Features | PCA sin interpretación | Ratios financieros + momentum |
| Insight | Solo predicción de precio | Score de valuación por empresa |
| Uso práctico | Bajo | Alto — identifica sub/sobrevaluación |

> **Nota sobre el R² negativo:** Un R² de -0.11 en predicción de retornos futuros es en realidad esperado y honesto — los mercados son semi-eficientes. El valor del modelo no está en predecir bien (nadie puede), sino en el *score de valuación relativa* que se obtiene del análisis de residuos.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

# Paleta
BG, BG2 = '#0D1117', '#161B22'
ACCENT, GREEN, RED, BLUE, GRAY, WHITE, GOLD = '#FF6B00','#2ECC71','#E74C3C','#3498DB','#8B949E','#F0F6FC','#F1C40F'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG2, 'axes.edgecolor': GRAY,
    'axes.labelcolor': WHITE, 'text.color': WHITE, 'xtick.color': GRAY, 'ytick.color': GRAY,
    'grid.color': '#21262D', 'grid.linestyle': '--', 'grid.alpha': 0.6, 'font.family': 'DejaVu Sans',
})
print('Setup completo')

## 1. Carga de datos

In [ ]:
BASE = 'https://raw.githubusercontent.com/nicolinolochex/DataScience/main/data/'
df_prices = pd.read_csv(BASE + 'prices-split-adjusted.csv', parse_dates=['date'])
df_sec    = pd.read_csv(BASE + 'securities.csv')
df_fund   = pd.read_csv(BASE + 'fundamentals.csv')

df_prices.rename(columns={'symbol': 'ticker'}, inplace=True)
df_sec.rename(columns={'Ticker symbol': 'ticker'}, inplace=True)
df_fund.rename(columns={'Ticker Symbol': 'ticker'}, inplace=True)
df_sec.drop(columns=['Date first added', 'SEC filings', 'CIK'], errors='ignore', inplace=True)
df_fund.drop(columns=['Unnamed: 0', 'For Year'], errors='ignore', inplace=True)

df_prices['month']       = df_prices['date'].dt.to_period('M')
df_prices['fiscal_year'] = df_prices['date'].dt.year
df_fund['Period Ending'] = pd.to_datetime(df_fund['Period Ending'])
df_fund['fiscal_year']   = df_fund['Period Ending'].dt.year

print(f'prices: {df_prices.shape} | sec: {df_sec.shape} | fund: {df_fund.shape}')

## 2. Modelo Original — Diagnóstico del Data Leakage

El modelo original usaba `close_mean`, `high_mean`, `low_mean` del mismo período como features para predecir `open_mean` del mismo período.
Esto introduce **data leakage directo**: el precio de apertura está perfectamente correlacionado con el precio de cierre del mismo día.

In [ ]:
# Construimos el dataset como en el modelo original
monthly = df_prices.groupby(['ticker', 'month']).agg(
    open_mean=('open', 'mean'), close_mean=('close', 'mean'),
    high_mean=('high', 'mean'), low_mean=('low', 'mean'), volume_sum=('volume', 'sum')
).reset_index()
monthly['fiscal_year'] = monthly['month'].dt.year

df_m1 = monthly.merge(df_sec[['ticker']], on='ticker', how='inner')
fund_dedup = df_fund.groupby(['ticker', 'fiscal_year'])[
    ['Earnings Per Share', 'Profit Margin', 'Operating Margin', 'After Tax ROE']
].mean().reset_index()
df_m1 = df_m1.merge(fund_dedup, on=['ticker', 'fiscal_year'], how='inner')

ORIG_FEATURES = ['close_mean', 'high_mean', 'low_mean', 'volume_sum',
                 'Earnings Per Share', 'Operating Margin', 'Profit Margin']
ORIG_TARGET   = 'open_mean'

df_orig = df_m1.dropna(subset=ORIG_FEATURES + [ORIG_TARGET]).copy()
X_orig  = StandardScaler().fit_transform(df_orig[ORIG_FEATURES].values)
y_orig  = df_orig[ORIG_TARGET].values

# Correlación entre open y close (el leakage)
corr = df_orig[['open_mean', 'close_mean']].corr().iloc[0, 1]
print(f'Correlacion open_mean vs close_mean (mismo período): {corr:.4f}')
print(f'=> R2 artificialmente inflado porque open ~ close del mismo mes')

In [ ]:
# Validación cruzada — ambos métodos sobre el mismo modelo original
rf_orig = RandomForestRegressor(n_estimators=200, max_depth=30, random_state=SEED, n_jobs=-1)

kf   = KFold(n_splits=3, shuffle=True, random_state=SEED)
tscv = TimeSeriesSplit(n_splits=5)

r2_orig_kfold = cross_val_score(rf_orig, X_orig, y_orig, cv=kf,   scoring='r2').mean()

df_orig_sorted = df_orig.sort_values('month')
X_orig_ts = StandardScaler().fit_transform(df_orig_sorted[ORIG_FEATURES].values)
y_orig_ts  = df_orig_sorted[ORIG_TARGET].values
r2_orig_ts = cross_val_score(rf_orig, X_orig_ts, y_orig_ts, cv=tscv, scoring='r2').mean()

print(f'Original KFold R2:           {r2_orig_kfold:.4f}  <- leakage')
print(f'Original TimeSeriesSplit R2: {r2_orig_ts:.4f}  <- mismo leakage, validacion correcta')

### Visualización: La Trampa del R²

El R²=0.9999 es una señal de alerta, no de éxito. Un modelo real con datos financieros nunca debería tener R² tan cercano a 1.

In [ ]:
from IPython.display import Image
Image('linkedin_assets/01_r2_trap.png')

## 3. Modelo Optimizado — Diseño correcto

### Cambios fundamentales:
1. **Target:** retorno forward a 3 meses `(price[t+3] - price[t]) / price[t]` — variable accionable y sin leakage
2. **Features:** fundamentales del año anterior (fy+1 join) + momentum de precios rezagados
3. **Validación:** `TimeSeriesSplit` que respeta la estructura temporal
4. **Modelo:** Gradient Boosting con regularización (`max_depth=4`, `subsample=0.8`)

In [ ]:
monthly2 = df_prices.groupby(['ticker', 'month']).agg(
    close_mean=('close', 'mean'), volume_sum=('volume', 'sum')
).reset_index()
monthly2['fiscal_year'] = monthly2['month'].dt.year
monthly2 = monthly2.sort_values(['ticker', 'month'])

# Target: retorno forward a 3 meses (SIN leakage — miramos el futuro como target, no como feature)
monthly2['close_t3']          = monthly2.groupby('ticker')['close_mean'].shift(-3)
monthly2['forward_return_3m'] = (monthly2['close_t3'] - monthly2['close_mean']) / monthly2['close_mean']

# Features: todo rezagado (no usamos datos del futuro)
monthly2['close_lag1']     = monthly2.groupby('ticker')['close_mean'].shift(1)
monthly2['price_momentum'] = monthly2.groupby('ticker')['close_mean'].pct_change(3)
monthly2['volume_lag1']    = monthly2.groupby('ticker')['volume_sum'].shift(1)

# Join fundamentales del AÑO ANTERIOR (evita leakage de reportes aun no publicados)
fund_slim = fund_dedup.copy()
fund_slim['fy_join'] = fund_slim['fiscal_year'] + 1
df_m2 = monthly2.merge(
    fund_slim.drop(columns='fiscal_year').rename(columns={'fy_join': 'fiscal_year'}),
    on=['ticker', 'fiscal_year'], how='inner'
)
df_m2 = df_m2.merge(df_sec[['ticker', 'GICS Sector', 'Security']], on='ticker', how='left')
df_m2 = df_m2.dropna(subset=['forward_return_3m'])
df_m2 = df_m2[df_m2['forward_return_3m'].between(-0.9, 2.0)]

OPT_FEATURES = ['Earnings Per Share', 'Profit Margin', 'Operating Margin',
                'After Tax ROE', 'close_lag1', 'price_momentum', 'volume_lag1']
df_opt = df_m2.dropna(subset=OPT_FEATURES + ['forward_return_3m']).sort_values('month').copy()

print(f'Dataset optimizado: {df_opt.shape}')
print(f'Rango temporal: {df_opt["month"].min()} — {df_opt["month"].max()}')
print(f'Empresas únicas: {df_opt["ticker"].nunique()}')

In [ ]:
X_opt = StandardScaler().fit_transform(df_opt[OPT_FEATURES].values)
y_opt = df_opt['forward_return_3m'].values

gb = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=SEED
)
r2_opt_ts = cross_val_score(gb, X_opt, y_opt, cv=tscv, scoring='r2').mean()
print(f'Optimizado TimeSeriesSplit R2: {r2_opt_ts:.4f}')
print()
print('Interpretacion:')
print(f'  R2={r2_opt_ts:.2f} es honesto — predecir retornos futuros es inherentemente difícil.')
print('  El valor no esta en predecir perfectamente, sino en el score de valuacion relativa')
print('  que obtenemos del análisis de residuos (retorno real vs predicho).')

## 4. Tabla comparativa completa

In [ ]:
Image('linkedin_assets/02_comparativa.png')

## 5. Feature Importance — ¿Qué variables realmente importan?

In [ ]:
# Entrenar sobre todos los datos para obtener importancias
gb.fit(X_opt, y_opt)

importance_df = pd.DataFrame({
    'Feature': OPT_FEATURES,
    'Importance': gb.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature importances (modelo optimizado):')
for _, row in importance_df.iterrows():
    bar = '=' * int(row['Importance'] * 100)
    print(f'  {row["Feature"]:<25} {bar} {row["Importance"]:.1%}')

In [ ]:
Image('linkedin_assets/03_feature_importance.png')

## 6. Nuevo Insight — Score de Valuación por Residuos

El insight más valioso del modelo optimizado no es la predicción en sí,
sino el **análisis de residuos**: cuando una empresa consistentemente genera un retorno
mayor al predicho por sus fundamentales, está **sistematicamente subvalorada** por el mercado.

In [ ]:
df_opt = df_opt.copy()
df_opt['predicted_return'] = gb.predict(X_opt)
df_opt['residual']         = df_opt['forward_return_3m'] - df_opt['predicted_return']

# Score de valuación: residuo promedio por empresa (mínimo 12 observaciones para robustez)
underval = df_opt.groupby(['ticker', 'GICS Sector']).agg(
    residual_mean=('residual', 'mean'),
    obs=('residual', 'count'),
    avg_actual=('forward_return_3m', 'mean')
).reset_index()
underval = underval[underval['obs'] >= 12]
underval['valuation'] = pd.cut(
    underval['residual_mean'],
    bins=[-np.inf, -0.02, 0.02, np.inf],
    labels=['Subvalorada', 'Valuacion justa', 'Sobrevaluada']
)

print('Distribucion de valuacion:')
print(underval['valuation'].value_counts())
print()

print('Top 10 empresas SUBVALORADAS (mayor residuo positivo del mercado):')
top_under = underval.nsmallest(10, 'residual_mean')
for _, r in top_under.iterrows():
    print(f"  {r['ticker']:<6} | {r['GICS Sector']:<30} | residuo: {r['residual_mean']:+.3f}")

print()
print('Top 10 empresas SOBREVALUADAS (retorno real menor al predicho):')
top_over = underval.nlargest(10, 'residual_mean')
for _, r in top_over.iterrows():
    print(f"  {r['ticker']:<6} | {r['GICS Sector']:<30} | residuo: {r['residual_mean']:+.3f}")

In [ ]:
Image('linkedin_assets/04_sector_valuation.png')

In [ ]:
Image('linkedin_assets/05_top_companies.png')

## 7. Conclusiones

### Lo que encontré revisando el modelo original:

1. **Data Leakage masivo:** El R²=0.9999 provenía de usar el precio de cierre del mismo período para predecir el precio de apertura. Una correlación trivial, no aprendizaje real.

2. **Validación temporal incorrecta:** KFold con `shuffle=True` mezcla datos de 2013 con datos de 2010, lo que equivale a "ver el futuro" durante el entrenamiento.

3. **Target sin sentido práctico:** Predecir el precio de apertura de hoy tiene cero valor financiero.

### Lo que el modelo optimizado agrega:

1. **Honestidad:** R²=-0.11 es la realidad de predecir retornos en mercados semi-eficientes.

2. **Valor accionable:** El score de valuación por residuos identifica empresas que sistemáticamente superan o quedan por debajo de lo esperado dado sus fundamentales.

3. **Insight sectorial:** El sector Energy aparece como el de mayor proporción de empresas subvaloradas (43%), coherente con el contexto 2010-2016 post-crisis de commodities.

4. **Features interpretables:** Sabemos qué impulsa el retorno (momentum + fundamentales del año anterior).

---
> *Análisis realizado sobre el dataset público del S&P 500 (2010-2016). No constituye asesoramiento financiero.*